# Collaborative Filtering Recommender

In this notebook, we build a collaborative filtering recommender using the MovieLens dataset.

Unlike non-personalized approaches, collaborative filtering generates user-specific recommendations by identifying similar users and using their rating patterns to predict missing ratings.

We start with a user-based collaborative filtering model using cosine similarity, and then compare it with Pearson correlation.

Contributors: Juan Múñoz, Santiago Llorca

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load data

In [2]:
try:
    ratings = pd.read_csv("ml-latest-small/ratings.csv")
    movies = pd.read_csv("ml-latest-small/movies.csv")
except FileNotFoundError:
    ratings = pd.read_csv("ratings.csv")
    movies = pd.read_csv("movies.csv")

## 2. Temporal Train-Test Split

We use a temporal split instead of a random split. This is more realistic for recommender systems because predictions should always be made using past interactions only.

In [3]:
ratings = ratings.sort_values("timestamp")

cutoff = int(len(ratings) * 0.8)
train = ratings.iloc[:cutoff].copy()
test = ratings.iloc[cutoff:].copy()

print("Train size:", len(train))
print("Test size:", len(test))

Train size: 80668
Test size: 20168


## 3. Build the User-Item Matrix

The user-item matrix is the foundation of collaborative filtering. Rows represent users, columns represent movies, and values are ratings.

In [4]:
user_item_matrix = train.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

print("User-item matrix shape:", user_item_matrix.shape)
user_item_matrix.head()

User-item matrix shape: (522, 7867)


movieId,1,2,3,4,5,6,7,8,9,10,...,146028,146210,146656,148424,148626,148675,149380,150548,152711,155168
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Baseline Model

We first compute the global mean rating. This is the simplest possible predictor and serves as a baseline to beat.

In [5]:
global_mean = train["rating"].mean()
print("Global mean rating:", round(global_mean, 4))

Global mean rating: 3.5085


## 5. User-Based Collaborative Filtering with Cosine Similarity

Cosine similarity compares users based on the angle between their rating vectors. It uses raw ratings and does not correct for user-specific rating scale bias.

In [6]:
user_item_filled = user_item_matrix.fillna(0)

user_similarity_cosine = cosine_similarity(user_item_filled)
user_similarity_cosine = pd.DataFrame(
    user_similarity_cosine,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

np.fill_diagonal(user_similarity_cosine.values, 0)

user_similarity_cosine.head()

userId,1,2,3,4,5,6,7,8,9,10,...,600,601,602,603,604,605,606,607,608,609
userId,,,,,,,,,,,,,,,,,,,,,
1,0.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.291272,0.072370,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572
2,0.027283,0.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.023248,0.094013,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565
3,0.059720,0.000000,0.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.013143,0.005113,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000
4,0.194395,0.003726,0.002251,0.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.211921,0.024891,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198
5,0.129080,0.016614,0.005020,0.128659,0.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.137053,0.097936,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232


## 6. Cosine Prediction Function

For each user-item pair, the model finds similar users who rated the target movie and computes a weighted average of their ratings.


In [7]:
def predict_rating_cosine(user_id, movie_id, user_item_matrix, similarity_matrix, k=20):
    # If movie not in training columns or user not in matrix, fallback to global mean
    if user_id not in user_item_matrix.index or movie_id not in user_item_matrix.columns:
        return np.nan

    # Users who rated the target movie
    movie_ratings = user_item_matrix[movie_id].dropna()

    if movie_ratings.empty:
        return np.nan

    # Similarities between target user and users who rated the movie
    sim_scores = similarity_matrix.loc[user_id, movie_ratings.index]

    # Keep top-k similar users
    top_k_users = sim_scores.sort_values(ascending=False).head(k)

    # Keep only positive similarities
    top_k_users = top_k_users[top_k_users > 0]

    if top_k_users.empty:
        return np.nan

    # Ratings of neighbors
    neighbor_ratings = movie_ratings.loc[top_k_users.index]

    # Weighted prediction
    pred = np.dot(top_k_users, neighbor_ratings) / top_k_users.sum()
    return pred

## 7. Generate Predictions with Cosine Similarity

In [8]:
test["pred_cosine"] = test.apply(
    lambda row: predict_rating_cosine(
        row["userId"],
        row["movieId"],
        user_item_matrix,
        user_similarity_cosine,
        k=20
    ),
    axis=1
)

test["pred_cosine"] = test["pred_cosine"].fillna(global_mean)
# Clip to valid rating range [0.5, 5.0]
test["pred_cosine"] = test["pred_cosine"].clip(0.5, 5.0)

test[["userId", "movieId", "rating", "pred_cosine"]].head()

,userId,movieId,rating,pred_cosine
79691,495,72998,5.0,3.755249
79564,495,2762,4.5,3.937078
79601,495,4993,0.5,4.302784
79709,495,89492,5.0,3.651498
79551,495,2028,4.5,4.115224


## 8. Evaluate Cosine Collaborative Filtering

In [9]:
mae_baseline = mean_absolute_error(test["rating"], [global_mean] * len(test))
rmse_baseline = np.sqrt(mean_squared_error(test["rating"], [global_mean] * len(test)))

mae_cosine = mean_absolute_error(test["rating"], test["pred_cosine"])
rmse_cosine = np.sqrt(mean_squared_error(test["rating"], test["pred_cosine"]))

print(f"Baseline MAE: {mae_baseline:.4f}")
print(f"Baseline RMSE: {rmse_baseline:.4f}")
print(f"Cosine CF MAE: {mae_cosine:.4f}")
print(f"Cosine CF RMSE: {rmse_cosine:.4f}")

Baseline MAE: 0.8540
Baseline RMSE: 1.0787
Cosine CF MAE: 0.8501
Cosine CF RMSE: 1.0742


Cosine collaborative filtering produces personalized predictions based on user-user similarity.

Its main limitation is sparsity. If similar users have not rated the target item, the model cannot generate a reliable prediction and must fall back to the global mean.

## 9. User-Based Collaborative Filtering with Pearson Correlation

Pearson correlation corrects for user rating bias by mean-centering each user's ratings before computing similarity.

This is useful because some users systematically rate higher or lower than others.

In [10]:
user_means = user_item_matrix.mean(axis=1)
user_item_centered = user_item_matrix.sub(user_means, axis=0)
user_item_centered_filled = user_item_centered.fillna(0)

user_similarity_pearson = cosine_similarity(user_item_centered_filled)
user_similarity_pearson = pd.DataFrame(
    user_similarity_pearson,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

np.fill_diagonal(user_similarity_pearson.values, 0)

user_similarity_pearson.head()

userId,1,2,3,4,5,6,7,8,9,10,...,600,601,602,603,604,605,606,607,608,609
userId,,,,,,,,,,,,,,,,,,,,,
1,0.000000,0.001265,0.000553,0.048419,0.021847,-0.045497,-0.006200,0.047013,0.019510,-0.008754,...,0.069528,0.025997,-0.017172,-0.015221,-0.037059,-0.029121,0.012016,0.055261,0.075224,-0.025713
2,0.001265,0.000000,0.000000,-0.017164,0.021796,-0.021051,-0.011114,-0.048085,0.000000,0.003012,...,-0.000980,-0.063099,-0.031581,-0.001688,0.000000,0.000000,0.006226,-0.020504,-0.006001,-0.060091
3,0.000553,0.000000,0.000000,-0.011260,-0.031539,0.004800,0.000000,-0.032471,0.000000,0.000000,...,-0.004669,-0.021902,-0.016117,0.017749,0.000000,-0.001431,-0.037289,-0.007789,-0.013001,0.000000
4,0.048419,-0.017164,-0.011260,0.000000,-0.029620,0.013956,0.058091,0.002065,-0.005874,0.051590,...,0.023748,-0.034784,0.063122,0.027640,-0.013782,0.040037,0.020590,0.014628,-0.037569,-0.017884
5,0.021847,0.021796,-0.031539,-0.029620,0.000000,0.009111,0.010117,-0.012284,0.000000,-0.033165,...,0.021435,0.024490,0.012427,0.027076,0.012461,-0.036272,0.026319,0.031896,-0.001751,0.093829


## 10. Pearson Prediction Function

In [11]:
def predict_rating_pearson(user_id, movie_id, user_item_matrix, centered_matrix, similarity_matrix, user_means, k=20):
    if user_id not in user_item_matrix.index or movie_id not in user_item_matrix.columns:
        return np.nan

    movie_ratings = user_item_matrix[movie_id].dropna()
    if movie_ratings.empty:
        return np.nan

    sim_scores = similarity_matrix.loc[user_id, movie_ratings.index]
    top_k_users = sim_scores.sort_values(ascending=False).head(k)
    top_k_users = top_k_users[top_k_users > 0]

    if top_k_users.empty:
        return np.nan

    centered_ratings = centered_matrix.loc[top_k_users.index, movie_id].dropna()

    aligned_users = top_k_users.index.intersection(centered_ratings.index)
    top_k_users = top_k_users.loc[aligned_users]
    centered_ratings = centered_ratings.loc[aligned_users]

    if top_k_users.empty:
        return np.nan

    pred = user_means.loc[user_id] + np.dot(top_k_users, centered_ratings) / top_k_users.sum()
    return pred

## 11. Generate Predictions with Pearson Correlation

In [12]:
test["pred_pearson"] = test.apply(
    lambda row: predict_rating_pearson(
        row["userId"],
        row["movieId"],
        user_item_matrix,
        user_item_centered,
        user_similarity_pearson,
        user_means,
        k=20
    ),
    axis=1
)

test["pred_pearson"] = test["pred_pearson"].fillna(global_mean)
# Clip to valid rating range [0.5, 5.0]
test["pred_pearson"] = test["pred_pearson"].clip(0.5, 5.0)

test[["userId", "movieId", "rating", "pred_pearson"]].head()

,userId,movieId,rating,pred_pearson
79691,495,72998,5.0,4.474270
79564,495,2762,4.5,4.915041
79601,495,4993,0.5,4.763956
79709,495,89492,5.0,4.792751
79551,495,2028,4.5,4.866406


## 12. Evaluate Pearson Collaborative Filtering

In [13]:
mae_pearson = mean_absolute_error(test["rating"], test["pred_pearson"])
rmse_pearson = np.sqrt(mean_squared_error(test["rating"], test["pred_pearson"]))

print(f"Pearson CF MAE: {mae_pearson:.4f}")
print(f"Pearson CF RMSE: {rmse_pearson:.4f}")

Pearson CF MAE: 0.8444
Pearson CF RMSE: 1.0714


## 13. Compare All Models

In [14]:
results_cf = pd.DataFrame({
    "Model": ["Global Mean", "Cosine CF", "Pearson CF"],
    "MAE": [mae_baseline, mae_cosine, mae_pearson],
    "RMSE": [rmse_baseline, rmse_cosine, rmse_pearson]
})

results_cf

,Model,MAE,RMSE
0,Global Mean,0.854019,1.078712
1,Cosine CF,0.850130,1.074241
2,Pearson CF,0.844439,1.071414


## 14. Interpretation of Results

The baseline predicts the same value for all user-item pairs, so it ignores any personalization.

Cosine collaborative filtering improves on this by leveraging similarity between users. Pearson collaborative filtering goes one step further by correcting for rating-scale differences across users.

In practice, Pearson is often more robust when users have different rating habits, while cosine is simpler and easier to compute.

## 15. Conclusion

In this notebook, we implemented user-based collaborative filtering using cosine similarity and Pearson correlation.

Both models provide personalized predictions and improve over a simple global mean baseline. However, collaborative filtering remains sensitive to sparsity, since predictions depend on overlap between users and items.

Overall, Pearson correlation is theoretically stronger because it adjusts for rating bias, while cosine similarity provides a simpler baseline for user-based collaborative filtering.

## 16. Export Predictions

We export the test set with all predictions so the evaluation notebook (05) can compute ranking metrics alongside the other recommenders.

In [15]:
# Export predictions for Notebook 5

pred_cf = test[["userId", "movieId", "rating"]].copy()

# Select the final collaborative filtering model
if "pred_pearson" in test.columns and "pred_cosine" in test.columns:
    if "rmse_pearson" in globals() and "rmse_cosine" in globals():
        best_cf_col = "pred_pearson" if rmse_pearson <= rmse_cosine else "pred_cosine"
    else:
        best_cf_col = "pred_pearson"
elif "pred_pearson" in test.columns:
    best_cf_col = "pred_pearson"
else:
    best_cf_col = "pred_cosine"

# Export a single score column for the final CF approach
pred_cf["score_cf"] = test[best_cf_col]

# Optional: keep the exact predicted rating column too
pred_cf["predicted_rating_cf"] = test[best_cf_col]

pred_cf.to_csv("predictions_cf.csv", index=False)

print("Exported collaborative filtering model:", best_cf_col)
print(pred_cf.shape)
pred_cf.head()

Exported collaborative filtering model: pred_pearson
(20168, 5)


,userId,movieId,rating,score_cf,predicted_rating_cf
79691,495,72998,5.0,4.474270,4.474270
79564,495,2762,4.5,4.915041,4.915041
79601,495,4993,0.5,4.763956,4.763956
79709,495,89492,5.0,4.792751,4.792751
79551,495,2028,4.5,4.866406,4.866406
